In [2]:
import os
import time
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

# Create required directories
os.makedirs('data', exist_ok=True)
os.makedirs('reports', exist_ok=True)

print("="*60)
print("PART 1: BASIC PARQUET OPERATIONS (EMPLOYEE DATASET)")
print("="*60)

# Task 1: Create DataFrame
employee_data = {
    "employee_id": [1, 2, 3, 4, 5],
    "name": ["Asha", "Rahul", "Neha", "Vikram", "Priya"],
    "department": ["IT", "HR", "IT", "Finance", "HR"],
    "salary": [60000, 45000, 70000, 55000, 48000]
}
employees_df = pd.DataFrame(employee_data)

# Task 2: Save as Parquet
employees_df.to_parquet('employees.parquet', index=False)
employees_df.to_parquet('data/employees.parquet', index=False)
print("✅ Saved 'employees.parquet' without index")

# Task 3: Read Parquet File
loaded_df = pd.read_parquet('employees.parquet')
print("\n--- Loaded Employee Records ---")
print(loaded_df)

# Task 4: Basic Analysis
high_salary_df = loaded_df[loaded_df['salary'] > 50000]
print("\n--- High Salary Employees (>50,000) ---")
print(high_salary_df)

print(f"\nAverage Salary: ${loaded_df['salary'].mean():,.2f}")
print("\nEmployee Counts by Department:")
print(loaded_df['department'].value_counts())

# Task 5: Save Filtered Data
high_salary_df.to_parquet('high_salary_employees.parquet', index=False)
high_salary_df.to_parquet('data/high_salary_employees.parquet', index=False)
print("\n✅ Saved 'high_salary_employees.parquet'")

# Bonus Task: Column Pruning
pruned_df = pd.read_parquet('employees.parquet', columns=['name', 'salary'])
print("\n--- Column Pruned Data (Name & Salary) ---")
print(pruned_df)


print("\n" + "="*60)
print("PART 2: APACHE PARQUET BENCHMARKING & COMPRESSION ANALYSIS")
print("="*60)

# Generate a larger synthetic dataset for meaningful benchmarking (100,000 rows)
np.random.seed(42)
n_rows = 100000

large_df = pd.DataFrame({
    'transaction_id': range(1, n_rows + 1),
    'user_id': np.random.randint(1000, 9999, size=n_rows),
    'category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Beauty', 'Sports'], size=n_rows),
    'amount': np.random.uniform(5.0, 500.0, size=n_rows).round(2),
    'year': np.random.choice([2023, 2024, 2025], size=n_rows),
    'region': np.random.choice(['North', 'South', 'East', 'West'], size=n_rows)
})

csv_path = 'data/large_transactions.csv'
large_df.to_csv(csv_path, index=False)
csv_size_mb = os.path.getsize(csv_path) / (1024 * 1024)

# Benchmarking Compression Codecs
codecs = ['snappy', 'gzip', 'zstd', 'none']
results = []

for codec in codecs:
    pq_path = f'data/transactions_{codec}.parquet'

    # Measure write time
    start_write = time.time()
    large_df.to_parquet(pq_path, index=False, compression=None if codec == 'none' else codec)
    write_time = time.time() - start_write

    # Measure read time
    start_read = time.time()
    _ = pd.read_parquet(pq_path)
    read_time = time.time() - start_read

    file_size_mb = os.path.getsize(pq_path) / (1024 * 1024)
    compression_ratio = ((csv_size_mb - file_size_mb) / csv_size_mb) * 100

    results.append({
        'Format/Codec': f'Parquet ({codec})',
        'File Size (MB)': round(file_size_mb, 2),
        'Write Time (s)': round(write_time, 4),
        'Read Time (s)': round(read_time, 4),
        'Space Saving vs CSV (%)': round(compression_ratio, 2)
    })

# Add CSV baseline to results
results.insert(0, {
    'Format/Codec': 'Raw CSV',
    'File Size (MB)': round(csv_size_mb, 2),
    'Write Time (s)': 'N/A',
    'Read Time (s)': 'N/A',
    'Space Saving vs CSV (%)': 0.0
})

benchmark_df = pd.DataFrame(results)
print("\n📊 Compression & Format Benchmarking Summary:")
print(benchmark_df.to_string(index=False))

# Measure Column Pruning Performance (CSV vs Parquet)
start_csv = time.time()
csv_pruned = pd.read_csv(csv_path, usecols=['category', 'amount'])
time_csv_pruned = time.time() - start_csv

start_pq = time.time()
pq_pruned = pd.read_parquet('data/transactions_snappy.parquet', columns=['category', 'amount'])
time_pq_pruned = time.time() - start_pq

print(f"\n⚡ Column Pruning Performance (Reading 2/6 columns):")
print(f"  - CSV Read Time:     {time_csv_pruned:.4f} sec")
print(f"  - Parquet Read Time: {time_pq_pruned:.4f} sec")
print(f"  - Speedup Factor:    {time_csv_pruned / time_pq_pruned:.2f}x faster!")

# Generate benchmark_results.md
with open('reports/benchmark_results.md', 'w') as f:
    f.write("# Task 10: Apache Parquet Performance & Compression Benchmark Report\n\n")
    f.write("## 1. File Size & Speed Comparison\n\n")
    f.write(benchmark_df.to_markdown(index=False))
    f.write(f"\n\n## 2. Key Findings\n")
    f.write(f"- **Column Pruning Speedup:** Parquet loaded selected columns **{time_csv_pruned / time_pq_pruned:.2f}x faster** than CSV.\n")
    f.write(f"- **Best Compression Codec:** `zstd` provided optimal balance between storage savings and read/write speed.\n")

print("\n✅ Saved 'reports/benchmark_results.md' successfully!")

PART 1: BASIC PARQUET OPERATIONS (EMPLOYEE DATASET)
✅ Saved 'employees.parquet' without index

--- Loaded Employee Records ---
   employee_id    name department  salary
0            1    Asha         IT   60000
1            2   Rahul         HR   45000
2            3    Neha         IT   70000
3            4  Vikram    Finance   55000
4            5   Priya         HR   48000

--- High Salary Employees (>50,000) ---
   employee_id    name department  salary
0            1    Asha         IT   60000
2            3    Neha         IT   70000
3            4  Vikram    Finance   55000

Average Salary: $55,600.00

Employee Counts by Department:
department
IT         2
HR         2
Finance    1
Name: count, dtype: int64

✅ Saved 'high_salary_employees.parquet'

--- Column Pruned Data (Name & Salary) ---
     name  salary
0    Asha   60000
1   Rahul   45000
2    Neha   70000
3  Vikram   55000
4   Priya   48000

PART 2: APACHE PARQUET BENCHMARKING & COMPRESSION ANALYSIS

📊 Compression & Format